In [ ]:
import pandas as pd
import shap
import optuna
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from lightgbm import LGBMRegressor
from sklearn.inspection import permutation_importance
import numpy as np
import talib as ta
from datetime import datetime, timezone
import config.config_binance as config
from backtest.backtester import run_backtest
from data_loader import fetch_historical_data, get_binance_client


In [ ]:
# Data loader
start_dt = datetime(2024, 7, 13, tzinfo=timezone.utc)
start_ms = int(start_dt.timestamp() * 1000)

end_dt   = datetime.now(tz=timezone.utc)
end_ms   = int(end_dt.timestamp() * 1000)
df = fetch_historical_data(
            symbol=config.TRADING_SYMBOL, interval="1m", start_str=start_ms, end_str=end_ms
        )
df

In [ ]:
data_lag = 120  # minute level data lag on binance
horizon = 60  # minutes interval after "now"
close_delayed = df["close"].shift(data_lag)
# log-return from the delayed price to (DELAY + H) minutes further out
df["log_ret_target"] = np.log(
    df["close"].shift(data_lag + horizon) / df["close"].shift(data_lag)
)

In [ ]:
# Simple lin. comb. prices
df['median'] = (df['high'] + df['low']) / 2
df['typical_price'] = (df['high'] + df['low'] + df['close']) / 3
df['weighted_close'] = (df['high'] + df['low'] + 2 * df['close']) / 4
df['ohlc_ave'] = (df['open'] + df['high'] + df['low'] + df['close']) / 4
df['high-low_range'] = df['high'] - df['low']
df['body_size'] = df['close'] - df['open']
df['upper_wick'] = df['high'] - np.maximum(df['open'], df['close'])
df['lower_wick'] = np.minimum(df['open'], df['close']) - df['low']

In [ ]:
# Return & momentum features
df["ret_1"] = df["close"].pct_change()
df["log_ret_1"] = np.log(df["close"]).diff()
for w in (5, 15, 30, 60):                                  # rolling windows
    df[f"ret_{w}"] = df["close"].pct_change(w)
    df[f"mom_{w}"] = df["close"].diff(w)                 # momentum
    df[f"roc_{w}"] = df["close"].pct_change(w) * 100     # rate of change %

In [ ]:
# Moving-average based signals
for w in (10, 20, 50, 200):
    df[f"sma_{w}"] = df["close"].rolling(w).mean()
    df[f"ema_{w}"] = df["close"].ewm(span=w, adjust=False).mean()
    df[f"close_above_sma_{w}"] = (df["close"] > df[f"sma_{w}"]).astype(int)
    df[f"distance_sma_{w}"] = df["close"] / df[f"sma_{w}"] - 1

In [ ]:
# Volatility metrics
df["hl_range"] = df["high"] - df["low"]
df["atr_14"] = df["hl_range"].rolling(14).mean()
df["rolling_std_30"] = df["ret_1"].rolling(30).std()
df["parkinson_vol_30"] = (
    (np.log(df["high"]/df["low"])**2).rolling(30).mean() * (1/(4*np.log(2)))
).pow(0.5)          # high-low-only estimator

In [ ]:
# Volume and Liquidity features
df["volume_rank_30"] = df["volume"].rank(
    pct=True, 
    method="max"
    ).rolling(30).apply(lambda x: x.iloc[-1])
df["dollar_vol"] = df["close"] * df["volume"]    # Filters out illiquid intervals
df["mean_trade_size"] = (
        df["volume"] / df["number_of_trades"].replace(0, np.nan)
    )
direction = np.sign(df["close"].diff()).fillna(0)          # -1, 0, +1
df["obv"] = (direction * df["volume"]).cumsum()
mfm = ((df["close"] - df["low"]) - (df["high"] - df["close"])) / (df["high"] - df["low"])  # money-flow
mfm = mfm.replace([np.inf, -np.inf], 0).fillna(0)          # handle hi==lo or NaN
df["ad_line"] = (mfm * df["volume"]).cumsum()

In [ ]:
# Time-based covariates
df["minute"] = df["open_time"].dt.minute
df["hour"] = df["open_time"].dt.hour
df["dayofweek"] = df["open_time"].dt.dayofweek
df["month"] = df["open_time"].dt.month
df["is_weekend"] = df["dayofweek"] >= 5
# cyclical encoding
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

In [ ]:
# Technical-indicator classics (non-linear combinations)
df["rsi_14"]  = ta.RSI(df["close"], timeperiod=14)
df["macd"], df["macd_signal"], df["macd_hist"] = ta.MACD(df["close"])
df["bb_upper"], df["bb_middle"], df["bb_lower"] = ta.BBANDS(df["close"])

In [ ]:
feature_cols = [c for c in df.columns if c != "log_ret_target"]
df_model = df.dropna(subset=["log_ret_target"])
X = df_model[feature_cols]
y = df_model["log_ret_target"]

print(X.columns)
non_numeric = X.select_dtypes(exclude=["number"]).columns
print("Non-numeric columns:", list(non_numeric))
X = X.drop(columns=non_numeric)
feature_cols = [c for c in X.columns]  # feature_cols update after non numeric cols drop
assert X.select_dtypes(exclude=["number"]).empty, "Still non-numeric cols!"

def predict_price(model, latest_df, delay=120, horizon=60):
    """
    latest_df : DataFrame that **already ends at the last available candle**
                (2 h ago if you just fetched it)
    """
    latest_close = latest_df["close"].iloc[-1]                 # P(t₀)
    x_live       = latest_df[feature_cols].iloc[-1:]           # 1-row frame
    pred_log_r   = model.predict(x_live)[0]                    # r̂ (from t₀ to t₀+2h+H)
    target_price = latest_close * np.exp(pred_log_r)           # P̂(t₀+2h+H)
    return target_price

# df_live   = fetch_historical_data(..., end_str="now UTC")      # ends at t₀
# limit_px  = predict_price(trained_model, df_live)

In [ ]:
# Feature selection checks
corr = X.assign(target=y).corr(method="spearman")   # rank-based = robust
plt.figure(figsize=(12,10))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(); plt.title("Spearman correlations"); plt.show()

In [ ]:
tscv   = TimeSeriesSplit(n_splits=5)     # chronological CV
params = dict(
    n_estimators=1000,
    learning_rate=0.01,
    subsample=0.7,
    colsample_bytree=0.8,
    max_depth=-1,
)
model_lgbm = LGBMRegressor(**params, verbosity=-1)

# single split for simplicity; wrap in loop if you like
train_idx, test_idx = list(tscv.split(X))[-1]
model_lgbm.fit(X.iloc[train_idx], y.iloc[train_idx])
y_pred = model_lgbm.predict(X.iloc[test_idx])
print("MAE:", np.mean(np.abs(y_pred - y.iloc[test_idx])))
print("RMSE:", np.sqrt(np.mean((y_pred - y.iloc[test_idx])**2))
      )
print("SMAPE:", np.mean(
    2*np.abs(y_pred - y.iloc[test_idx])/(np.abs(y_pred)+np.abs(y.iloc[test_idx])))
    )

In [ ]:
model = model_lgbm

In [ ]:
# Model agnostic permutative importance
perm = permutation_importance(
    model, X.iloc[test_idx], y.iloc[test_idx],
    n_repeats=10, random_state=42, scoring="neg_mean_absolute_error"
)
pi = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)
print(pi.head(15))

In [ ]:
# Tree explainer SHAP values
expl = shap.TreeExplainer(model)
sh_values = expl.shap_values(X.iloc[test_idx])
shap.summary_plot(sh_values, X.iloc[test_idx], max_display=25)

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample", 0.5, 1.0),
    }
    # model = LGBMRegressor(**params,)
    mae_scores = []
    for train_idx, test_idx in TimeSeriesSplit(n_splits=5).split(X):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        y_pred = model.predict(X.iloc[test_idx])
        mae_scores.append(mean_absolute_error(y.iloc[test_idx], y_pred))
    return np.mean(mae_scores)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)
print("Best:", study.best_params)

In [ ]:
def walk_forward(model, X, y, train_size=0.7, step=10_000):
    """
    Walk-forward evaluation that retrains every `step` rows and tests on the
    next `step` (or whatever is left at the end).

    Returns
    -------
    preds   : numpy.ndarray  concatenated predictions
    actuals : numpy.ndarray  matching ground truth
    """
    preds, actuals = [], []
    train_end = int(len(X) * train_size)

    while train_end < len(X):                       # ①  test *while* we still have data
        test_end = min(train_end + step, len(X))    # ②  clamp to frame length

        # --- fit on everything *before* the test slice --------------------
        model.fit(X.iloc[:train_end], y.iloc[:train_end])

        # --- predict the current out-of-sample block ----------------------
        y_hat = model.predict(X.iloc[train_end:test_end])
        preds.extend(y_hat)
        actuals.extend(y.iloc[train_end:test_end])

        print(f"train_end → {train_end}  test rows: {test_end-train_end}")

        train_end = test_end                        # ③  slide window forward

    return np.array(preds), np.array(actuals)

# model = LGBMRegressor(**study.best_params)
step = 10_000
p, a = walk_forward(model, X, y, step=step)

In [ ]:
def slice_offsets(preds, actuals, step, offsets):
    """
    Collect y_pred and y_true at the same *offset* inside every test block.

    preds, actuals : 1-D arrays returned by walk_forward
    step           : same step you passed to walk_forward
    offsets        : list/tuple of integers (0-based offsets inside a block)

    returns dict offset → (y_true_slice, y_pred_slice)
    """
    out = {}
    n_blocks = len(preds) // step
    for off in offsets:
        idx = np.arange(off, off + n_blocks*step, step)
        out[off] = (actuals[idx], preds[idx])
    return out

mae  = mean_absolute_error(a, p)              # a = actuals, p = preds
rmse = root_mean_squared_error(a, p)

print(f"MAE  (log-return points): {mae:.6f}")
print(f"RMSE (log-return points): {rmse:.6f}")
print(f"MAE  in bp             : {mae*1e4:.2f}")
print(f"RMSE in bp             : {rmse*1e4:.2f}")


In [ ]:
offsets = [0, 1, 3, 9, 29, 59]          # 1st, 2nd, 4th, 10th, 30th, 60th
slices  = slice_offsets(p, a, step, offsets)

for off, (yy, pp) in slices.items():
    print(f"\nOffset {off} (minute {off+1} of each chunk)")
    print("  rows :", len(yy))
    print("  MAE  :", mean_absolute_error(yy, pp))
    print("  RMSE :", root_mean_squared_error(yy, pp))

In [ ]:
df.to_parquet(f"../data/{config.TRADING_SYMBOL}_20240713_20250713.parquet")

In [ ]:
# latest_close   = df["close"].iloc[-1]          # P_t
# pred_log_ret   = model.predict(X_live)[0]      # ŷ = r̂
# target_price   = latest_close * np.exp(pred_log_ret)